Thai Legal CCL - RAG Querying Pipeline
Hybrid Search (Dense + Sparse + RRF Fusion + Reranker) with LangChain

### Import

In [1]:
import torch
from typing import List, Tuple

from FlagEmbedding import FlagReranker, BGEM3FlagModel
from qdrant_client import QdrantClient
from qdrant_client.models import SparseVector, Prefetch, Fusion, FusionQuery

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# --- เลือก LLM ที่ต้องการใช้ (uncomment อันที่ต้องการ) ---
# from langchain_openai import ChatOpenAI
# from langchain_anthropic import ChatAnthropic
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.chat_models import ChatOllama

print("✅ Imports OK")

d:\Github_ThisPC\muict_thai_legal_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports OK


### Configuration

In [2]:
# ─── Qdrant ───────────────────────────────────────────────
QDRANT_URL      = "http://localhost:6333"
COLLECTION_NAME = "thai_laws_collection"
VECTOR_DENSE    = "dense"
VECTOR_SPARSE   = "sparse"

# ─── Models ───────────────────────────────────────────────
EMBED_MODEL_NAME = "VISAI-AI/nitibench-ccl-human-finetuned-bge-m3"
RERANK_MODEL     = "BAAI/bge-reranker-v2-m3"

# ─── Retrieval ────────────────────────────────────────────
RETRIEVAL_LIMIT = 3   # จำนวน candidates ที่ส่งให้ reranker
FINAL_LIMIT     = 3    # top-k ที่ส่งให้ LLM
LLM_MODEL_NAME = "scb10x/typhoon2.5-qwen3-4b"

print("✅ Config OK")

✅ Config OK


### System Propmt

In [3]:
SYSTEM_PROMPT = """คุณคือผู้ช่วยด้านกฎหมายไทย (Thai Legal AI Assistant) ที่มีความเชี่ยวชาญด้านกฎหมายแพ่งและพาณิชย์ (CCL)
คุณจะตอบคำถามโดยอิงจากข้อความกฎหมายที่ถูกดึงมาเท่านั้น ห้ามอนุมานหรือแต่งเติมข้อมูลที่ไม่มีในบริบท

## กฎการตอบ
1. อ้างอิง **ชื่อกฎหมาย** และ **มาตรา** ที่เกี่ยวข้องทุกครั้ง
2. หากบริบทไม่มีข้อมูลเพียงพอ ให้แจ้งว่า "ไม่พบข้อมูลที่ตรงกับคำถามในฐานข้อมูลกฎหมายที่มีอยู่"
3. ตอบเป็นภาษาไทยที่ชัดเจน กระชับ และเข้าใจง่าย
4. หากมีโทษทางอาญา ให้ระบุอัตราโทษอย่างครบถ้วน (จำคุก / ปรับ / ทั้งจำทั้งปรับ)
5. ห้ามให้คำแนะนำเชิงกลยุทธ์ทางกฎหมาย — แนะนำให้ปรึกษาทนายความสำหรับคดีจริง

## รูปแบบการตอบ
**สรุปคำตอบ:** [ตอบโดยตรง 1-2 ประโยค]

**มาตราที่เกี่ยวข้อง:**
- [ชื่อกฎหมาย มาตรา X]: [สาระสำคัญ]

**หมายเหตุ:** [ข้อควรระวัง หรือข้อจำกัดของคำตอบ ถ้ามี]

---
## บริบทจากฐานข้อมูลกฎหมาย
{context}
"""

print("✅ System Prompt OK")

✅ System Prompt OK


### Initialize Models & Client

In [4]:
def _init_device() -> Tuple[str, bool]:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    use_fp16 = device == "cuda"
    print(f"[RAG] Running on: {device.upper()}")
    return device, use_fp16


def build_models() -> Tuple[BGEM3FlagModel, FlagReranker, QdrantClient]:
    """Initialize embedding model, reranker, and Qdrant client."""
    device, use_fp16 = _init_device()

    embed_model = BGEM3FlagModel(
        EMBED_MODEL_NAME,
        use_fp16=use_fp16,
        batch_size=16,
        device=device,
    )

    reranker = FlagReranker(
        RERANK_MODEL,
        use_fp16=True,
        devices=device,
    )

    qdrant_client = QdrantClient(url=QDRANT_URL)

    return embed_model, reranker, qdrant_client

print("✅ Models & Qdrant client ready")

✅ Models & Qdrant client ready


### HybridRetriever Class

In [5]:
class HybridRetriever:
    """
    Retriever ที่รวม Dense + Sparse search ด้วย RRF Fusion
    และ Rerank ด้วย BGE-Reranker-v2-M3
    """

    def __init__(
        self,
        embed_model: BGEM3FlagModel,
        reranker: FlagReranker,
        client: QdrantClient,
        retrieval_limit: int = RETRIEVAL_LIMIT,
        final_limit: int = FINAL_LIMIT,
    ) -> None:
        self.embed_model     = embed_model
        self.reranker        = reranker
        self.client          = client
        self.retrieval_limit = retrieval_limit
        self.final_limit     = final_limit

    def _encode(self, query: str) -> Tuple[list, SparseVector]:
        """แปลง query เป็น dense vector และ sparse vector"""
        out = self.embed_model.encode(
            [query],
            return_dense=True,
            return_sparse=True,
            max_length=512,
        )
        dense_vec  = out["dense_vecs"][0].tolist()
        sparse_dict = out["lexical_weights"][0]
        sparse_vec = SparseVector(
            indices=[int(k)   for k in sparse_dict.keys()],
            values =[float(v) for v in sparse_dict.values()],
        )
        return dense_vec, sparse_vec

    def _search(self, dense_vec: list, sparse_vec: SparseVector) -> list:
        """Hybrid search ใน Qdrant ด้วย RRF Fusion"""
        return self.client.query_points(
            collection_name=COLLECTION_NAME,
            prefetch=[
                Prefetch(query=dense_vec,  using=VECTOR_DENSE,  limit=self.retrieval_limit),
                Prefetch(query=sparse_vec, using=VECTOR_SPARSE, limit=self.retrieval_limit),
            ],
            query=FusionQuery(fusion=Fusion.RRF),
            limit=self.retrieval_limit,
            with_payload=True,
        ).points

    def _rerank(self, query: str, candidates: list) -> list:
        """Rerank candidates ด้วย BGE-Reranker แล้วคืน top final_limit"""
        if not candidates:
            return []
        pairs  = [[query, r.payload["text"]] for r in candidates]
        scores = self.reranker.compute_score(pairs, batch_size=32)
        for i, r in enumerate(candidates):
            r.score = float(scores[i])
        candidates.sort(key=lambda x: x.score, reverse=True)
        return candidates[: self.final_limit]

    def retrieve(self, query: str) -> List[Document]:
        """Pipeline หลัก: query → encode → search → rerank → Documents"""
        dense_vec, sparse_vec = self._encode(query)
        candidates            = self._search(dense_vec, sparse_vec)
        ranked                = self._rerank(query, candidates)
        return [
            Document(
                page_content=r.payload.get("text", ""),
                metadata={
                    "law_name":    r.payload.get("law_name", ""),
                    "section_num": r.payload.get("section_num", ""),
                    "score":       r.score,
                },
            )
            for r in ranked
        ]

    # ให้เรียกเป็น callable ได้ (LangChain RunnableLambda)
    def __call__(self, query: str) -> List[Document]:
        return self.retrieve(query)


print("✅ HybridRetriever defined")

✅ HybridRetriever defined


### Context Formatter & RAG Chain

In [6]:
def format_context(docs: List[Document]) -> str:
    """แปลง Documents → structured string สำหรับใส่ใน prompt"""
    if not docs:
        return "ไม่พบข้อมูลกฎหมายที่เกี่ยวข้อง"
    parts = [
        f"[{i}] {doc.metadata.get('law_name', '')} มาตรา {doc.metadata.get('section_num', '')}\n{doc.page_content}"
        for i, doc in enumerate(docs, start=1)
    ]
    return "\n\n".join(parts)


def build_rag_chain(retriever: HybridRetriever, llm):
    """สร้าง LangChain LCEL chain สำหรับ RAG"""
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human",  "{question}"),
    ])

    # sub-chain: question → retrieve → format
    context_chain = (
        RunnableLambda(lambda x: x["question"])
        | RunnableLambda(retriever.retrieve)
        | RunnableLambda(format_context)
    )

    # full chain
    return (
        RunnablePassthrough.assign(context=context_chain)
        | prompt
        | llm
        | StrOutputParser()
    )


print("✅ format_context & build_rag_chain defined")

✅ format_context & build_rag_chain defined


### Initialize LLM & Build RAG Chain

In [7]:
class ThaiLegalRAG:
    """High-level chat interface for Thai Legal RAG."""

    def __init__(
        self,
        llm,
        retrieval_limit: int = RETRIEVAL_LIMIT,
        final_limit: int = FINAL_LIMIT,
    ) -> None:
        embed_model, reranker, client = build_models()

        self.retriever = HybridRetriever(
            embed_model=embed_model,
            reranker=reranker,
            client=client,
            retrieval_limit=retrieval_limit,
            final_limit=final_limit,
        )
        self.chain = build_rag_chain(self.retriever, llm)

    def chat(self, query: str) -> str:
        """
        Accept a user query and return the LLM-generated answer.

        Args:
            query: คำถามภาษาไทยเกี่ยวกับกฎหมาย

        Returns:
            คำตอบที่อ้างอิงจากฐานข้อมูลกฎหมาย
        """
        return self.chain.invoke({"question": query})

    def chat_with_sources(self, query: str) -> Tuple[str, List[Document]]:
        """
        Same as chat() but also returns the source documents for inspection.

        Returns:
            (answer, retrieved_docs)
        """
        docs = self.retriever.retrieve(query)
        context = format_context(docs)

        from langchain_core.prompts import ChatPromptTemplate

        prompt = ChatPromptTemplate.from_messages(
            [("system", SYSTEM_PROMPT), ("human", "{question}")]
        )
        llm_step = self.chain.steps[-2]
        chain = (
            RunnablePassthrough.assign(context=lambda _: context)
            | prompt
            | llm_step
            | StrOutputParser()
        )
        answer = chain.invoke({"question": query, "context": context})
        return answer, docs

    def debug(self, query: str):
        # 1. Encode
        dense_vec, sparse_vec = self.retriever._encode(query)

        # 2. Retrieve (before rerank)
        candidates = self.retriever._search(dense_vec, sparse_vec)

        # 3. Rerank
        reranked = self.retriever._rerank(query, candidates)

        # 4. Convert to readable docs
        docs = []
        for r in reranked:
            p = r.payload
            docs.append(
                {
                    "text": p.get("text", ""),
                    "law_name": p.get("law_name", ""),
                    "section": p.get("section_num", ""),
                    "score": r.score,
                }
            )

        # 5. Build context
        context = format_context(
            [Document(page_content=d["text"], metadata=d) for d in docs]
        )

        # ❗ IMPORTANT: อย่าเรียก self.chat() (จะ retrieve ซ้ำ)
        answer = self.chain.invoke({"question": query, "context": context})

        return {
            "query": query,
            "num_candidates": len(candidates),
            "num_final": len(reranked),
            "reranked": docs,
            "context": context,
            "answer": answer,
        }

In [8]:
import os

# --- ตั้ง API Key (เลือกอันที่ใช้) ---
os.environ["OPENAI_API_KEY"] = "ollama"          # OpenAI
# os.environ["ANTHROPIC_API_KEY"]  = "sk-ant-..." # Anthropic
# os.environ["GOOGLE_API_KEY"]     = "AIza..."    # Google

# --- เลือก LLM ---
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm = ChatOllama(model=LLM_MODEL_NAME, temperature=0)
# llm = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0)
# llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

# --- Build RAG ---
rag = ThaiLegalRAG(llm=llm, retrieval_limit=RETRIEVAL_LIMIT, final_limit=FINAL_LIMIT)
print("✅ RAG chain ready")

C:\Users\User\AppData\Local\Temp\ipykernel_19684\4232178010.py:10: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model=LLM_MODEL_NAME, temperature=0)


[RAG] Running on: CUDA


Fetching 23 files: 100%|██████████| 23/23 [00:00<00:00, 44476.25it/s]


✅ RAG chain ready


In [10]:
question = "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร"
answer = rag.chat(question)
print(answer)

**สรุปคำตอบ:** ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน

**มาตราที่เกี่ยวข้อง:**
- พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132: ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน

**หมายเหตุ:** ข้อความกฎหมายระบุเฉพาะกรณีที่ไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 จึงต้องใช้มาตรา 132 เท่านั้น หากไม่ระบุว่าเป็นกรณีใดกรณีหนึ่งเพิ่มเติม จะไม่สามารถอนุมานเพิ่มเติมได้


In [ ]:
question = "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร"
answer, sources = rag.chat_with_sources(question)

print("=" * 60)
print("📌 ANSWER")
print("=" * 60)
print(answer)

print("\n" + "=" * 60)
print("📚 RETRIEVED SOURCES")
print("=" * 60)
for doc in sources:
    m = doc.metadata
    print(f"  [{m['law_name']} มาตรา {m['section_num']}]  score={m['score']:.4f}")
    print(f"  {doc.page_content[:120]}...\n")